In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
columns = []
for i in range(1,6):
    columns.append(f'Building {i}')

In [3]:
time = np.arange(0,24,1)

In [4]:
priceSwitch = [(13,15,19,24), (10, 19, 28, 10)]
priceDaily = []
for i in range(0,len(time)):
    for j in range(0,len(time)//24):
        for k in range(0,len(priceSwitch[0])):
            if i < priceSwitch[0][k]:
                priceDaily.append(priceSwitch[1][k])
                break

In [5]:
flexUsage = np.zeros(len(columns))
usageTime = np.zeros(len(columns))

Testing of community optim

In [2]:
from community_optim import *

In [3]:
with open('transInfo.json') as fp:
    transInfo = json.load(fp)

In [4]:
controlAliasList = ["2", "3", "4", "5", "9", "10", "12", "15", "17", "22", "24", "25", "26", "27", "28"]

In [ ]:
with open('../../configs/indexMapping.json') as fp:
    sensorIdxMapping = json.load(fp)        # Map sensor indices to simulation indices
simIdxMapping = {v: k for k, v in sensorIdxMapping.items()}     # Map simulation indices to sensor indices
aliasesSensorIdx = [simIdxMapping[alias] for alias in controlAliasList]       # Convert list of controlled buildings from sim idx to sensor idx

In [5]:
n = 10
numBuildings = len(controlAliasList)
# coordinator = Coordinator(len(controlAliasList), n, 1)
coordinator = Coordinator(2, n, 1)

Is DPP? True
Is DCP? True


In [6]:
coordinator.AdjustInit(verbose=True)

(CVXPY) Feb 18 09:41:48 PM: Your problem has 20 variables, 70 constraints, and 112 parameters.
(CVXPY) Feb 18 09:41:48 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 18 09:41:48 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 18 09:41:48 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 18 09:41:48 PM: Compiling problem (target solver=CLARABEL).
(CVXPY) Feb 18 09:41:48 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> CLARABEL
(CVXPY) Feb 18 09:41:48 PM: Applying reduction Dcp2Cone
(CVXPY) Feb 18 09:41:48 PM: Applying reduction CvxAttr2Constr
(CVXPY) Feb 18 09:41:48 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Feb 18 09:41:48 PM: Applying reduction CLARABEL
(CVXPY) Feb 18 09:41:48 PM: Finished problem compilation (took 2.669e-02 seconds).
(CVXPY) Feb 18 09:41:48 PM: (Subsequent compilations of this problem, using the same arguments, shou

                                     CVXPY                                     
                                     v1.7.5                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------
           Clarabel.rs v0.11.1  -  Clever Acronym                

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 30
  constraints 

(CVXPY) Feb 18 09:41:48 PM: Problem status: optimal
(CVXPY) Feb 18 09:41:48 PM: Optimal value: 1.298e-09
(CVXPY) Feb 18 09:41:48 PM: Compilation took 2.669e-02 seconds
(CVXPY) Feb 18 09:41:48 PM: Solver (including time spent in interface) took 2.924e-03 seconds


  0  +1.0178e+04  -4.0107e+04  4.94e+00  3.06e-01  1.65e+00  1.00e+00  7.83e+03   ------   
  1  +8.9913e+01  -2.6796e+04  2.99e+02  1.87e-02  7.77e-02  7.11e-01  2.88e+02  9.74e-01  
  2  +9.8603e+00  -1.5472e+03  1.58e+02  1.08e-03  1.00e-01  4.41e+01  4.48e+01  9.57e-01  
  3  +7.6507e+00  -4.1262e+01  6.39e+00  3.37e-05  5.41e-02  1.20e+00  1.49e+00  9.67e-01  
  4  +1.2861e+00  -7.3898e+00  6.75e+00  2.34e-06  8.85e-03  1.13e-01  2.38e-01  9.90e-01  
  5  +2.5058e-01  -6.5499e-01  9.06e-01  2.95e-08  2.89e-04  7.78e-03  2.22e-02  9.90e-01  
  6  +4.8728e-02  -1.1733e-01  1.66e-01  3.47e-10  4.34e-06  1.29e-03  3.06e-03  9.90e-01  
  7  +9.6684e-03  -2.2029e-02  3.17e-02  4.07e-12  5.07e-08  2.32e-04  4.28e-04  9.90e-01  
  8  +1.9447e-03  -4.2688e-03  6.21e-03  4.85e-14  5.96e-10  4.36e-05  6.06e-05  9.90e-01  
  9  +3.9483e-04  -8.4477e-04  1.24e-03  7.12e-16  7.01e-12  8.43e-06  8.65e-06  9.90e-01  
 10  +8.0682e-05  -1.6955e-04  2.50e-04  1.47e-16  7.64e-14  1.66e-06  1.24e-06 

In [7]:
params = coordinator.adjustProb.prob.param_dict
vars = coordinator.adjustProb.prob.var_dict

In [8]:
for key, value in params.items():
    print(key, value.value.shape, value.value.sum())

usagePenalty (2,) 0.2
predLoad (2, 10) 20.0
flexMax (2, 10) 4000.0
flexMin (2, 10) 0.0
baseLoad (2, 10) 20.0
pow_ref (10,) 40.0
trans_ref (2, 10) 40.0


In [9]:
for key, value in vars.items():
    print(key, value.value.shape, value.value.sum())

flexLoad (2, 10) -0.00015797101411384206


In [10]:
vars['flexLoad'].value

array([[-7.89855071e-06, -7.89855071e-06, -7.89855071e-06,
        -7.89855071e-06, -7.89855071e-06, -7.89855071e-06,
        -7.89855071e-06, -7.89855071e-06, -7.89855071e-06,
        -7.89855071e-06],
       [-7.89855071e-06, -7.89855071e-06, -7.89855071e-06,
        -7.89855071e-06, -7.89855071e-06, -7.89855071e-06,
        -7.89855071e-06, -7.89855071e-06, -7.89855071e-06,
        -7.89855071e-06]])

In [11]:
flex = vars['flexLoad'].value
# flex = np.zeros((2, n))

In [13]:
print(2*(params['usagePenalty'].value@flex)**2@np.ones(n))
print(1*np.max(flex, axis=0)@np.ones(n))
print(3*np.ones(2)@np.power(flex, 2)@np.ones(n))

0.005410318561530746
-0.8223684212026526
0.4057738921148058


In [ ]:
coordinator.TransformerOverload()
coordinator.overloadList

True


[array([-50., -50., -50., -50., -50., -50., -50., -50., -50., -50., -50.,
        -50., -50., -50., -50., -50., -50., -50., -50., -50., -50., -50.,
        -50., -50., -50., -50., -50., -50., -50., -50., -50., -50., -50.,
        -50., -50., -50., -50., -50., -50., -50., -50., -50., -50., -50.,
        -50., -50., -50., -50., -50., -50., -50., -50., -50., -50., -50.,
        -50., -50., -50., -50., -50.]),
 array([-100., -100., -100., -100., -100., -100., -100., -100., -100.,
        -100., -100., -100., -100., -100., -100., -100., -100., -100.,
        -100., -100., -100., -100., -100., -100., -100., -100., -100.,
        -100., -100., -100., -100., -100., -100., -100., -100., -100.,
        -100., -100., -100., -100., -100., -100., -100., -100., -100.,
        -100., -100., -100., -100., -100., -100., -100., -100., -100.,
        -100., -100., -100., -100., -100., -100.])]

In [ ]:
coordinator.predictedLoad = predictedLoad
coordinator.predictedFlexibility = predictedFlex
coordinator.baseLoad = np.ones(coordinator.nsteps) * 10
overload = coordinator.TransformerOverload()
adjustValues = coordinator.Adjust(verbose=True)
coordinator.Dispatch(adjustValues)

In [23]:
flexLoad = cp.Variable((5,2), name='flexLoad')
objective = cp.Minimize(cp.sum_squares(flexLoad))
constraints = [flexLoad <= np.ones((5,2)) * 5,
               flexLoad >= np.ones((5,2))*2]

prob = cp.Problem(objective, constraints)
prob.solve()

40.000000000000014

In [24]:
flexLoad.value

array([[2., 2.],
       [2., 2.],
       [2., 2.],
       [2., 2.],
       [2., 2.]])